
# RADBIO_NEURO_002 — Biology Calibration Layer

This notebook takes the physics-layer output from `RADBIO_NEURO_001` and builds the first defensible biology calibration layer.

The central correction is that ROS is **not** hardcoded to suppress firing. The model separates:

- fast ROS/channel redox effects, which can transiently increase excitability;
- slow mitochondrial/ATP failure, which suppresses excitability and synaptic plasticity under chronic exposure.

Current outputs are a calibration scaffold. Replace approximate validation ratios after digitizing the primary paper figures/tables.


In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from calibrated_ros_mito_model import *
DATA = ROOT / 'data'
OUT = ROOT / 'outputs'
FIG = ROOT / 'figures'
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

## 1. Load physics forcing and validation targets

In [ ]:
forcing = pd.read_csv(DATA / 'biology_ready_dose_forcing_table.csv')
targets = pd.read_csv(DATA / 'validation_targets.csv')
provenance = pd.read_csv(DATA / 'parameter_provenance.csv')
display(forcing.head())
display(targets[['target_id','model_target_variable','dose_Gy','dose_rate_mGy_day','target_ratio_to_control','calibration_role','status']])

## 2. Calibrate the scaffold model to current numeric fit anchors

In [ ]:
params = calibrate_to_targets(targets, n_random=5000, seed=7)
print(params)
print('score =', score_params(params, targets))
with open(OUT / 'calibrated_parameters.json', 'w') as f:
    json.dump(params_to_dict(params), f, indent=2)

## 3. Model vs validation targets

In [ ]:
fit_rows = []
for _, row in targets.iterrows():
    if 'fit_anchor' not in str(row.get('calibration_role', '')):
        continue
    sim = simulate_constant_exposure(float(row['dose_rate_mGy_day']), int(row['exposure_days']), params)
    pred = float(sim[row['model_target_variable']].iloc[-1])
    fit_rows.append({
        'target_id': row['target_id'],
        'model_target_variable': row['model_target_variable'],
        'target_ratio_to_control': row['target_ratio_to_control'],
        'target_uncertainty_1sigma': row['target_uncertainty_1sigma'],
        'model_prediction': pred,
        'absolute_error': abs(pred - row['target_ratio_to_control']),
        'status': row['status'],
    })
fit_df = pd.DataFrame(fit_rows)
fit_df.to_csv(OUT / 'model_vs_validation_targets.csv', index=False)
display(fit_df)
ax = fit_df.plot(x='target_id', y=['target_ratio_to_control','model_prediction'], kind='bar', figsize=(10,5))
ax.set_ylabel('Ratio to control')
ax.set_xlabel('Validation target')
ax.set_title('Model vs current validation targets')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(FIG / 'model_vs_validation_targets.png', dpi=200)
plt.show()

## 4. Apply calibrated scaffold to SPENVIS LEO and Van Allen forcing table

In [ ]:
predictions, timecourses = predict_for_forcing_table(forcing, params)
predictions.to_csv(OUT / 'biology_calibrated_endpoint_predictions.csv', index=False)
timecourses.to_csv(OUT / 'biology_calibrated_daily_timecourses_all_scenarios.csv', index=False)
selected = predictions[predictions['shield_mm_Al'].isin([1,2,5,10,20])].copy()
selected.to_csv(OUT / 'selected_shielding_biology_summary.csv', index=False)
display(selected[['scenario','shield_mm_Al','dose_rate_mGy_day','mito_integrity_day_end','excitability_ratio_day_end','ltp_proxy_day_end','model_status']])

## 5. Biology endpoint plots versus shielding

In [ ]:
sel = predictions[predictions['shield_mm_Al'].isin([1,2,5,10,20])]
for metric, ylabel, fname in [
    ('mito_integrity_day_end','Mitochondrial integrity at day 180','mito_integrity_vs_shielding.png'),
    ('excitability_ratio_day_end','Excitability ratio at day 180','excitability_vs_shielding.png'),
    ('ltp_proxy_day_end','LTP proxy at day 180','ltp_proxy_vs_shielding.png'),
]:
    pivot = sel.pivot(index='shield_mm_Al', columns='scenario', values=metric).sort_index()
    ax = pivot.plot(marker='o', figsize=(8,5))
    ax.set_xscale('log')
    ax.set_xlabel('Al shielding depth (mm)')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel + ' vs shielding')
    plt.tight_layout()
    plt.savefig(FIG / fname, dpi=200)
    plt.show()

## 6. Timecourses at 10 mm Al

In [ ]:
subset = timecourses[timecourses['shield_mm_Al'].round(6).eq(10.0)]
for metric, ylabel, fname in [
    ('ros_norm','ROS state','timecourse_ros_10mm.png'),
    ('mito_integrity','Mitochondrial integrity','timecourse_mito_10mm.png'),
    ('excitability_ratio','Excitability ratio','timecourse_excitability_10mm.png'),
]:
    fig, ax = plt.subplots(figsize=(8,5))
    for scenario, grp in subset.groupby('scenario'):
        ax.plot(grp['day'], grp[metric], label=scenario)
    ax.set_xlabel('Mission day')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel + ' timecourse at 10 mm Al')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG / fname, dpi=200)
    plt.show()

## 7. Interpretation flags

- Use `biology_calibrated_endpoint_predictions.csv` as a model-development output only.
- Do not present it as astronaut medical risk.
- The next required improvement is primary-figure digitization and target refitting.
- The next engineering layer is `RADBIO_NEURO_003`: Brian2/NEURON conductance-level coupling.